# 05 — Strategy Optimizer

Finds the optimal pit stop strategy for a driver/race by simulating every valid 1-stop and 2-stop strategy using the trained LightGBM model.

**Sections:**
1. Load model + build base features for a specific driver/race
2. Run optimizer → top 10 strategies
3. Lap-by-lap chart of the best strategy
4. Undercut scenario vs a nominated rival

In [ ]:
import sys, os
sys.path.append("..")
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import importlib
import warnings
warnings.filterwarnings("ignore")

import src.model as model_mod
import src.strategy as strat_mod
importlib.reload(model_mod)
importlib.reload(strat_mod)
from src.strategy import (
    build_base_features, optimise, check_undercut,
    get_driver_skill, get_fp2_features, get_quali_features,
    compute_tyre_profiles, get_tyre_info,
)
from src.model import load_model
from src.pitstop_stats import load_pitstop_stats

model       = load_model()
pit_stats   = load_pitstop_stats()

# Build tyre profiles from historical race data (run once — saves to CSV)
tyre_profiles = compute_tyre_profiles()
print("Model, pit stats and tyre profiles loaded.")

## 1. Configure race and driver

Edit the variables below to change the driver, race, or conditions.

In [ ]:
# ── Race configuration ────────────────────────────────────────────────────
DRIVER       = "VER"
EVENT_NAME   = "British Grand Prix"
YEAR         = 2024
ROUND_NUMBER = 12
TOTAL_LAPS   = 52

AVAILABLE_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]
TRACK_TEMP          = 42.0      # °C

# ── Tyre profile for this track (auto-derived from historical data) ────────
tyre_info = get_tyre_info(EVENT_NAME, tyre_profiles)
print(f"Historical starting compounds : {tyre_info['starting_compounds']}")
print(f"Max stint lengths             : {tyre_info['max_stint']}")

# ── Auto-fill from CSVs ───────────────────────────────────────────────────
fp2   = get_fp2_features(DRIVER, YEAR, ROUND_NUMBER)
quali = get_quali_features(DRIVER, YEAR, ROUND_NUMBER)
skill = get_driver_skill(DRIVER)

# Cap teammate gap — removes anomalous sessions (crashes/DNFs in quali)
quali["QualiTeammateGap"] = float(np.clip(quali.get("QualiTeammateGap", 0), -1.5, 1.5))

print(f"\nDriver       : {DRIVER}")
print(f"Event        : {EVENT_NAME} {YEAR}")
print(f"FP2 pace     : {fp2['FP2LongRunPace']:.3f}s")
print(f"FP2 deg rate : {fp2['FP2DegRate']:.4f}s/lap")
print(f"Quali pos    : P{int(quali['QualiPosition'])}")
print(f"Gap to pole  : {quali['GapToPole']:.3f}s")
print(f"Driver skill : {skill:.3f}")

In [ ]:
# Load team performance index and degradation rate from laps_features.csv
laps = pd.read_csv("../data/processed/laps_features.csv")

race_laps = laps[
    (laps["Driver"] == DRIVER) &
    (laps["Year"] == YEAR) &
    (laps["RoundNumber"] == ROUND_NUMBER)
]

team_perf = float(race_laps["TeamPerformanceIndex"].median()) if not race_laps.empty else 0.0
deg_rate  = float(race_laps["DegradationRate"].median())      if not race_laps.empty else 0.08
rolling   = float(race_laps["RollingSeasonForm"].median())    if not race_laps.empty else 0.0
tcs       = float(race_laps["TrackCharacterScore"].median())  if not race_laps.empty else 0.0

print(f"TeamPerformanceIndex  : {team_perf:.4f}")
print(f"DegradationRate       : {deg_rate:.4f}")
print(f"RollingSeasonForm     : {rolling:.4f}")
print(f"TrackCharacterScore   : {tcs:.4f}")

In [ ]:
base = build_base_features(
    driver               = DRIVER,
    event_name           = EVENT_NAME,
    year                 = YEAR,
    fp2_long_run_pace    = fp2["FP2LongRunPace"],
    fp2_deg_rate         = fp2["FP2DegRate"],
    quali_position       = int(quali["QualiPosition"]),
    gap_to_pole          = quali["GapToPole"],
    quali_teammate_gap   = quali["QualiTeammateGap"],
    driver_skill_score   = skill,
    rolling_season_form  = rolling,
    track_character_score= tcs,
    team_performance_index = team_perf,
    degradation_rate     = deg_rate,
    track_temp           = TRACK_TEMP,
)
print("Base features built.")
print(pd.Series(base))

## 2. Run optimizer — top 10 strategies

In [ ]:
strategies = optimise(
    base_features        = base,
    total_laps           = TOTAL_LAPS,
    available_compounds  = AVAILABLE_COMPOUNDS,
    event_name           = EVENT_NAME,
    model                = model,
    pit_stats_df         = pit_stats,
    max_stops            = 2,
    top_n                = 10,
    step                 = 3,
    # starting_compounds and max_stint_by_compound are auto-loaded
    # from tyre_profiles.csv — override here if needed:
    # starting_compounds   = ["MEDIUM"],
    # max_stint_by_compound = {"SOFT": 20, "MEDIUM": 30, "HARD": 40},
)

print(f"\nTop {len(strategies)} strategies for {DRIVER} — {EVENT_NAME} {YEAR}")
print(f"(Starting compounds: {tyre_info['starting_compounds']}, Max stints: {tyre_info['max_stint']})\n")
print(f"{'Rank':<5} {'Strategy':<30} {'Pit Laps':<20} {'Stops':<7} {'Total Time'}")
print("-" * 78)
for i, r in enumerate(strategies, 1):
    compounds_str = " → ".join(r.compounds)
    pits_str      = ", ".join(f"L{p}" for p in r.pit_laps)
    print(f"  {i:<4} {compounds_str:<30} {pits_str:<20} {r.stops:<7} {r.total_time_str}")

## 3. Lap-by-lap breakdown — best strategy

Bar chart of predicted lap times coloured by compound (F1 standard colours: red = SOFT, yellow = MEDIUM, white/grey = HARD).

In [ ]:
best = strategies[0]
print("Best strategy:", best.summary())

COMPOUND_COLOURS = {"SOFT": "#E8002D", "MEDIUM": "#FFF200", "HARD": "#CACACA"}

fig = go.Figure()
for compound in best.compounds:
    sub = best.lap_times[best.lap_times["Compound"] == compound]
    fig.add_trace(go.Bar(
        x=sub["Lap"],
        y=sub["PredictedLapTime"],
        name=compound,
        marker_color=COMPOUND_COLOURS.get(compound, "steelblue"),
    ))

# Pit stop lap markers
for pit_lap in best.pit_laps:
    fig.add_vline(x=pit_lap + 0.5, line_dash="dash", line_color="white",
                  annotation_text=f"Pit L{pit_lap}",
                  annotation_font_color="white")

fig.update_layout(
    title=f"{DRIVER} — {EVENT_NAME} {YEAR} — Best Strategy: {' → '.join(best.compounds)}",
    xaxis_title="Lap",
    yaxis_title="Predicted Lap Time (s)",
    barmode="stack",
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
    font_color="white",
    legend_title="Compound",
    height=480,
)
fig.show()

In [ ]:
# Tyre degradation curve — lap time vs tyre life per compound
fig2 = px.line(
    best.lap_times,
    x="TyreLife",
    y="PredictedLapTime",
    color="Compound",
    color_discrete_map=COMPOUND_COLOURS,
    markers=True,
    title=f"Tyre Degradation Curve — {DRIVER} {EVENT_NAME} {YEAR}",
    labels={"TyreLife": "Tyre Age (laps)", "PredictedLapTime": "Lap Time (s)"},
    height=400,
)
fig2.update_layout(
    plot_bgcolor="#1a1a2e", paper_bgcolor="#1a1a2e", font_color="white"
)
fig2.show()

## 4. Undercut scenario

Given the current race state, should we pit now to undercut a rival?

In [ ]:
# ── Rival configuration ───────────────────────────────────────────────────
RIVAL_DRIVER        = "NOR"
CURRENT_LAP         = 25

# Your current state
YOUR_COMPOUND       = "MEDIUM"
YOUR_TYRE_LIFE      = 18
NEW_COMPOUND        = "HARD"     # compound you would fit
GAP_TO_RIVAL        = 1.8        # seconds (positive = you are behind)

# Rival's current state
RIVAL_COMPOUND      = "MEDIUM"
RIVAL_TYRE_LIFE     = 20
RIVAL_PLANNED_PIT   = 30         # lap rival is expected to pit

# Rival base features
rival_fp2   = get_fp2_features(RIVAL_DRIVER, YEAR, ROUND_NUMBER)
rival_quali = get_quali_features(RIVAL_DRIVER, YEAR, ROUND_NUMBER)
rival_skill = get_driver_skill(RIVAL_DRIVER)

rival_laps = laps[
    (laps["Driver"] == RIVAL_DRIVER) &
    (laps["Year"] == YEAR) &
    (laps["RoundNumber"] == ROUND_NUMBER)
]
rival_team_perf = float(rival_laps["TeamPerformanceIndex"].median()) if not rival_laps.empty else 0.0
rival_deg_rate  = float(rival_laps["DegradationRate"].median())      if not rival_laps.empty else 0.08
rival_rolling   = float(rival_laps["RollingSeasonForm"].median())    if not rival_laps.empty else 0.0
rival_tcs       = float(rival_laps["TrackCharacterScore"].median())  if not rival_laps.empty else 0.0

rival_base = build_base_features(
    driver                 = RIVAL_DRIVER,
    event_name             = EVENT_NAME,
    year                   = YEAR,
    fp2_long_run_pace      = rival_fp2["FP2LongRunPace"],
    fp2_deg_rate           = rival_fp2["FP2DegRate"],
    quali_position         = int(rival_quali["QualiPosition"]),
    gap_to_pole            = rival_quali["GapToPole"],
    quali_teammate_gap     = rival_quali["QualiTeammateGap"],
    driver_skill_score     = rival_skill,
    rolling_season_form    = rival_rolling,
    track_character_score  = rival_tcs,
    team_performance_index = rival_team_perf,
    degradation_rate       = rival_deg_rate,
    track_temp             = TRACK_TEMP,
)
print(f"Rival base features built for {RIVAL_DRIVER}.")

In [ ]:
result = check_undercut(
    your_base          = base,
    rival_base         = rival_base,
    current_lap        = CURRENT_LAP,
    your_compound      = YOUR_COMPOUND,
    your_tyre_life     = YOUR_TYRE_LIFE,
    rival_compound     = RIVAL_COMPOUND,
    rival_tyre_life    = RIVAL_TYRE_LIFE,
    rival_planned_pit  = RIVAL_PLANNED_PIT,
    new_compound       = NEW_COMPOUND,
    total_laps         = TOTAL_LAPS,
    event_name         = EVENT_NAME,
    gap_to_rival       = GAP_TO_RIVAL,
    model              = model,
    pit_stats_df       = pit_stats,
)

verdict = "✅ UNDERCUT RECOMMENDED" if result["undercut_possible"] else "❌ Undercut unlikely to work"
print(f"\n{verdict}")
print(f"  Starting gap to {RIVAL_DRIVER}   : {GAP_TO_RIVAL:+.3f}s (positive = you behind)")
print(f"  Projected gap at {RIVAL_DRIVER} pit exit: {result['gap_at_rival_exit']:+.3f}s (positive = you ahead)")
print(f"  Your total time over window     : {result['your_time_to_exit']:.3f}s")
print(f"  Rival total time over window    : {result['rival_time_to_exit']:.3f}s")
if result["recommended_pit_lap"]:
    print(f"  Recommended: pit on lap {result['recommended_pit_lap']}")

In [ ]:
lap_by_lap = result["lap_by_lap"]
if not lap_by_lap.empty:
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(
        x=lap_by_lap["Lap"], y=lap_by_lap["YourLapTime"],
        name=f"{DRIVER} (undercut)",
        line=dict(color="#00bfff", width=2),
        mode="lines+markers",
    ))
    fig3.add_trace(go.Scatter(
        x=lap_by_lap["Lap"], y=lap_by_lap["RivalLapTime"],
        name=f"{RIVAL_DRIVER} (stay out)",
        line=dict(color="#ff7043", width=2),
        mode="lines+markers",
    ))
    fig3.add_vline(
        x=CURRENT_LAP + 0.5, line_dash="dash", line_color="#00bfff",
        annotation_text=f"{DRIVER} pits", annotation_font_color="#00bfff"
    )
    fig3.add_vline(
        x=RIVAL_PLANNED_PIT + 0.5, line_dash="dash", line_color="#ff7043",
        annotation_text=f"{RIVAL_DRIVER} pits", annotation_font_color="#ff7043"
    )
    fig3.update_layout(
        title=f"Undercut Simulation: {DRIVER} vs {RIVAL_DRIVER} — {EVENT_NAME} {YEAR}",
        xaxis_title="Lap",
        yaxis_title="Lap Time (s)",
        plot_bgcolor="#1a1a2e",
        paper_bgcolor="#1a1a2e",
        font_color="white",
        height=420,
    )
    fig3.show()

    # Cumulative gap delta
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(
        x=lap_by_lap["Lap"], y=lap_by_lap["GapDelta"],
        name="Cumulative gap gained vs rival",
        fill="tozeroy",
        line=dict(color="#00e676"),
        mode="lines",
    ))
    fig4.add_hline(y=GAP_TO_RIVAL, line_dash="dot", line_color="white",
                   annotation_text="Gap needed to overcut",
                   annotation_font_color="white")
    fig4.add_hline(y=0, line_color="grey")
    fig4.update_layout(
        title="Cumulative Time Gained by Undercutting",
        xaxis_title="Lap",
        yaxis_title="Seconds gained vs rival",
        plot_bgcolor="#1a1a2e",
        paper_bgcolor="#1a1a2e",
        font_color="white",
        height=350,
    )
    fig4.show()